# 03: Cross-architecture comparison

**What this shows.** Per-layer linear fit of CKA vs SNR for MUSE, MP-SENet, and Demucs, plotted as **slope vs intercept**. Slope is *adaptivity* (how much CKA changes per dB), intercept is *robustness* (CKA at 0 dB). This is the qualitative shape of Fig. 5 in the poster.

**What to look for.** Each architecture forms its own cluster with a negative slope-intercept correlation: the more robust a layer, the less adaptive — and vice versa. Demucs sits at the highly-adaptive / lowly-robust corner; MUSE has the widest spread; MP-SENet is between.

**Runtime.** <30 s on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, MODEL_LABELS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## Load all three models

We restrict to each model's natural probe-layer set: norm1 layers for MUSE and MP-SENet, block-output layers for Demucs. The demo parquets already contain only the probed layers; we just normalise the model-name field for downstream grouping.

In [ ]:
def _path(model):
    full = DATA_DIR / f"snr/cka_snr_{model}.parquet"
    return full if full.exists() else DATA_DIR / f"cka_snr_{model}_demo.parquet"

def _probe_layers(model, layers):
    if model == "demucs":
        return list(layers)
    return [L for L in layers if L.endswith(".norm1")]

frames = []
for model in ["muse", "mpsenet", "demucs"]:
    df = pd.read_parquet(_path(model), columns=["clean_idx", "snr", "layer", "CKA", "model_name"])
    df["model_name"] = model
    df = df[df["layer"].isin(_probe_layers(model, df["layer"].unique()))]
    frames.append(df)
    print(f"{MODEL_LABELS[model]}: {len(df):,} rows, {df['layer'].nunique()} probe layers")
all_snr = pd.concat(frames, ignore_index=True)

## Per-layer linear fit: CKA vs SNR

For each (model, layer) we average CKA across utterances/noise and fit `CKA = slope * SNR + intercept`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

snr_avg = all_snr.groupby(["model_name", "layer", "snr"], as_index=False)["CKA"].mean()

rows = []
for (model, layer), g in snr_avg.groupby(["model_name", "layer"]):
    g = g.sort_values("snr")
    x, y = g["snr"].to_numpy(dtype=float), g["CKA"].to_numpy(dtype=float)
    if len(np.unique(x)) < 2:
        continue
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    rows.append({"model": model, "layer": layer,
                  "slope": reg.coef_[0], "intercept": reg.intercept_,
                  "r2": reg.score(x.reshape(-1, 1), y)})
trend = pd.DataFrame(rows)
print(f"trend rows: {len(trend)} (" + ", ".join(f"{m}={(trend['model']==m).sum()}" for m in ["muse","mpsenet","demucs"]) + ")")
trend.groupby("model")[["slope", "intercept", "r2"]].agg(["mean", "median"]).round(3)

## Slope vs Intercept (poster Fig. 5)

One marker per probe layer; one regression line per architecture.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

fig, ax = plt.subplots(figsize=(6.5, 4.8))
for model in ["muse", "mpsenet", "demucs"]:
    sub = trend[trend["model"] == model]
    if len(sub) < 2:
        continue
    r_val, _ = stats.pearsonr(sub["slope"], sub["intercept"])
    rho_val, _ = stats.spearmanr(sub["slope"], sub["intercept"])
    ax.scatter(sub["intercept"], sub["slope"],
               c=MODEL_COLORS[model], s=42, alpha=0.75,
               edgecolors="white", linewidths=0.4,
               label=fr"{MODEL_LABELS[model]}  ($r$={r_val:.2f}, $\rho$={rho_val:.2f})",
               zorder=3)
    coeffs = np.polyfit(sub["intercept"], sub["slope"], 1)
    xs = np.linspace(sub["intercept"].min(), sub["intercept"].max(), 100)
    ax.plot(xs, np.polyval(coeffs, xs),
            color=MODEL_COLORS[model], linestyle="--", linewidth=1.4, alpha=0.7, zorder=2)

ax.set_xlabel(r"Intercept  (CKA at 0 dB $\to$ robustness)")
ax.set_ylabel(r"Slope  ($\Delta$CKA per dB $\to$ adaptivity)")
ax.set_title("Robustness vs adaptivity across architectures")
ax.grid(True, alpha=0.15, zorder=0)
ax.legend(loc="best", framealpha=0.9)
plt.tight_layout();